# 05 – Puolueluokittelija, erottuvuus ja monimuotoisuus

**Tehtävä:** opetetaan malli, joka arvaa yksittäisen täysistuntopuheen perusteella puhujan puolueen. Mallin ennusteista lasketaan
- **polarisaatio** (*polarisation*): kuinka hyvin puolueet erottuvat toisistaan
- **puolueen sisäinen monimuotoisuus** (*within-party diversity*): kuinka paljon saman puolueen edustajat eroavat toisistaan

**Syöte:** `Data/processed/counts.npz` ja `counts_index.parquet` (muistio 04)
**Tulokset** kansioon `results/` (luetaan muistioissa 07 ja 08). Jokaisesta ajosta syntyy viisi tiedostoa, joiden nimi alkaa ajon tunnuksella (kahden luokan ajosta, kuten `government`, vain `_summary`):
- `<tunnus>_summary.csv`: tarkkuus, tasapainotettu tarkkuus, macro-F1 ja polarisaatio per kausi ja toisto
- `<tunnus>_pairwise.csv`: puolueparien erottuvuus
- `<tunnus>_confusion.json`: sekaannusmatriisi
- `<tunnus>_mp_profiles.csv`: edustajien profiilit (`person_id`, ei nimiä)
- `<tunnus>_diversity.csv`: puolueiden monimuotoisuus

Ajoista `party` ja `party_nostyle` tallennetaan lisäksi jokaisen puheen testiennusteet (*out-of-fold predictions*) tiedostoon `Data/processed/<tunnus>_oof.parquet`. Muistio 11 laskee niistä erottuvuuden teemoittain. Tiedosto on suuri (noin 20 Mt), joten se on johdettua aineistoa eikä kuulu versionhallintaan.

**Ajot:**

| tunnus | mitä | miksi |
|---|---|---|
| `party` | 8 puoluetta, kaikki kaudet | päätulos |
| `government` | hallitus vs. oppositio | kontrollitehtävä: kuinka paljon puolue-eroista selittyy hallitusasemalla? |
| `party_nominister` | 8 puoluetta ilman ministerien puheita | ministerit puhuvat hallituksen edustajina |
| `party_blue` | 9 puoluetta, Siniset mukana, vain 2015–19 | Sinisten sijainti |
| `party_nostyle` | 8 puoluetta ilman tyylisanoja ja jääneitä puolueiden nimiä | onko erottuvuus vain puhetapaa? |

**Kesto:** kaikki ajot yhteensä 15–55 minuuttia koneesta riippuen. Valmiit tulokset luetaan kansiosta `results/`, joten ajoja ei tarvitse toistaa; uudelleenlaskenta tehdään asetuksella `UUDELLEEN = True`. Nopeaa kokeilua varten `TOISTOT` ja `SATUNNAISET` voi asettaa pienemmiksi.

## Menetelmä

Käsitteet ja niiden englanninkieliset vastineet ovat seuraavan osion taulukossa.

**Malli.** TF-IDF-painotus ja multinomiaalinen logistinen regressio (scikit-learn). Jokaisella sanalla ja sanaparilla on paino jokaiselle puolueelle; puheen painot lasketaan yhteen ja muutetaan todennäköisyyksiksi, joiden summa on 1. TF-IDF pienentää hyvin yleisten sanojen painoa, ja `sublinear_tf` vaimentaa toistoa (sama sana kymmenen kertaa ei paina kymmentä kertaa enemmän). Regularisointi (C = 2) estää yksittäisiä harvinaisia sanoja saamasta suurta painoa.

**Harhojen torjunta.** Gentzkow ym. (2019) varoittavat, että luokittelijan tarkkuus voi näyttää puolue-eroja silloinkin, kun niitä ei ole (s. 1310, Peterson & Spirling), ja että pieni aineisto yliarvioi erot (*finite-sample bias*, s. 1308 ja 1314–1315). Siksi:

1. **Jako edustajittain.** Kauden edustajat jaetaan viiteen ryhmään (`StratifiedGroupKFold`). Malli opetetaan neljällä ryhmällä ja testataan viidennellä, ja vuoro kiertää. Sama edustaja ei ole koskaan sekä opetus- että testiaineistossa, joten malli ei voi oppia tunnistamaan yksittäisiä ihmisiä (pienissä puolueissa kolme edustajaa pitää jopa 70 % puheista, ks. EDA). Simola ym. varoittavat, että yksittäisten edustajien maneerit nostavat erottuvuutta (s. 19).
2. **Tasapainotettu opetusaineisto.** Jokainen puolue saa `N_OPETUS = 300` opetuspuhetta joka osiossa ja joka kaudella. Muuten malli arvaisi isoja puolueita useammin, ja kausi, jolla on enemmän aineistoa, näyttäisi polarisoituneemmalta.
3. **Toistot ja osaotanta** (*subsampling*). Ajo toistetaan 8 kertaa, joka kerta satunnaisella 80 %:lla kunkin puolueen edustajista. Toistojen hajonta on epävarmuusväli. Tavallinen bootstrap ei sovi, koska sama edustaja voisi päätyä sekä opetukseen että testiin; osaotanta vastaa Gentzkowin ym. menettelyä (s. 1321, kuvan 2 selite).
4. **Satunnaistesti** (*permutation test*; Gentzkowin ym. *random series*). Ajo toistetaan 4 kertaa niin, että puoluenimikkeet on sekoitettu edustajien kesken (Gentzkow ym. s. 1318; Simola ym. s. 19). Tarkkuuden pitää silloin olla arvaustasolla ja erottuvuuden noin 0.
5. **Sekoittavat tekijät** (*confounders*). Gentzkow ym. (s. 1316) ja Simola ym. (s. 13–14 ja 19–21) vakioivat hallitusaseman, alueen ja sukupuolen. Alue on tärkeä, koska pöytäkirjoissa säilyy alueellinen puhetapa (Simola ym. s. 9) ja aluevakiointi vaikuttaa juuri täytesanoihin, kuten *sit tämmöis* ja *elik tääl* (s. 20). Me tarkistamme hallitusaseman kontrollitehtävällä (ajo 2) ja tyylisanat ajolla 5. Alue- ja sukupuolivakiointi on tekemättä.

**Mittarit** lasketaan testiennusteista (*out-of-fold predictions*) eli puheista, joita malli ei ole nähnyt. Mallin tavalliset mittarit ovat tarkkuus (*accuracy*), tasapainotettu tarkkuus (*balanced accuracy*), macro-F1 ja sekaannusmatriisi (*confusion matrix*); ne on määritelty käsitetaulukossa. Tutkimuksen omat mittarit:
- **Erottuvuus** puolueiden A ja B välillä: d = 2·AUC − 1. AUC on todennäköisyys, että satunnainen A:n puhe saa suuremman pistemäärän log p(A) − log p(B) kuin satunnainen B:n puhe. 0 = puolueet eivät erotu, 1 = erottuvat aina. Mittari ei riipu luokittelurajasta, eikä se ole yhtä herkkä puolueiden määrälle kuin tarkkuus.
- **Polarisaatio** = erottuvuuden keskiarvo kaikkien 28 puolueparin yli.
- **Edustajan profiili** = mallin todennäköisyyksien keskiarvo 20 satunnaisesta puheesta. Kiinteä puhemäärä pitää satunnaisvaihtelun samana kaikille edustajille. Profiili lasketaan kahdesti eri puheista (A ja B), joten mukana ovat vain edustajat, joilla on vähintään 40 puhetta.
- **Monimuotoisuus** = profiilien kohinakorjattu hajonta puolueen keskipisteen ympärillä: √(keskiarvo (Aᵢ − Ā)·(Bᵢ − B̄)). A:n ja B:n satunnaisvaihtelu on riippumatonta, joten se kumoutuu keskimäärin (sama leave-out-ajatus kuin Gentzkowilla ym.). Yhdestä profiilista laskettu naiivi hajonta olisi kohinan takia liian suuri.
- **Suhteellinen monimuotoisuus** = monimuotoisuus jaettuna puolueiden keskipisteiden keskimääräisellä etäisyydellä. Se erottaa kaksi asiaa: tulevatko edustajat samankaltaisemmiksi vai erkaantuvatko puolueet toisistaan.

## Käsitteet ja mittarit

Raportissa käytetään englanninkielisiä termejä, jotka ovat taulukossa suluissa. Muut muistiot viittaavat tähän taulukkoon.

**Aineisto ja piirteet**

| käsite | englanniksi | merkitys |
|---|---|---|
| kausi | parliamentary term | vaalikausi: 2015–19, 2019–23, 2023–27 (aineistossa syyskuuhun 2026) |
| analyysiotos | analysis sample | puheet, jotka jäävät muistion 02 rajausten jälkeen |
| vartalointi | stemming | sanan päätteen katkaisu, esim. *hallituksen* → *hallituks* |
| lemmatisointi | lemmatisation | sanan palautus perusmuotoonsa, esim. *hallituksen* → *hallitus* |
| täytesana | stop word | yleinen sana, jolla ei ole omaa sisältöä (*ja*, *on*, *että*) |
| sanapari | bigram | kaksi peräkkäistä vartaloa, esim. `ulkomaalais_maanomistus` |
| fraasi | phrase, term | sanaston alkio: sana tai sanapari |
| piirre | feature | mallin syöte; tässä fraasin painotettu esiintymismäärä puheessa |
| piirrematriisi X | feature matrix | puheet × fraasit -matriisi, jossa arvot ovat fraasien esiintymismääriä; koodissa nimellä `X` alan tavan mukaan |
| harva matriisi | sparse matrix | matriisi, johon tallennetaan vain nollasta poikkeavat arvot |
| TF-IDF | term frequency–inverse document frequency | painotus, joka kasvattaa puheessa esiintyvän fraasin painoa ja pienentää kaikkialla esiintyvien fraasien painoa |

**Malli ja validointi**

| käsite | englanniksi | merkitys |
|---|---|---|
| luokittelija | classifier | malli, joka arvaa puheen luokan (tässä puolueen) |
| multinomiaalinen logistinen regressio | multinomial logistic regression | lineaarinen luokittelija, joka antaa jokaiselle puolueelle todennäköisyyden |
| regularisointi | regularisation | painojen kutistaminen ylisovittamisen estämiseksi; C on regularisoinnin käänteinen voimakkuus |
| opetus- ja testiaineisto | training / test set | aineisto, jolla malli opetetaan, ja aineisto, jolla se arvioidaan |
| ristiinvalidointi edustajittain | grouped cross-validation | aineisto jaetaan osioihin niin, että edustajan kaikki puheet ovat samassa osiossa |
| osio | fold | yksi ristiinvalidoinnin viidestä osasta |
| tasapainotettu opetusaineisto | balanced training set | jokaiselta puolueelta sama määrä opetuspuheita |
| testiennuste | out-of-fold prediction | ennuste puheelle, jota malli ei nähnyt opetuksessa |
| kontrollitehtävä | control task | sama malli, mutta arvattavana hallitus vs. oppositio |
| osaotanta | subsampling | toistuva otanta ilman takaisinpanoa (tässä 80 % edustajista) |
| epävarmuusväli | uncertainty interval | toistojen 10. ja 90. persentiilin väli |
| satunnaistesti | permutation test | luokat sekoitetaan edustajien kesken ja analyysi toistetaan; mittaa menetelmän harhaa |
| satunnaissarja | random series | satunnaistestin tulokset (Gentzkow ym.) |

**Luokittelijan mittarit** (lasketaan testiennusteista)

| käsite | englanniksi | merkitys |
|---|---|---|
| tarkkuus | accuracy | osuus puheista, joiden puolue arvattiin oikein; isot puolueet painavat enemmän |
| saanti | recall | puolueen puheista oikein arvattujen osuus |
| täsmällisyys | precision | puolueelle arvatuista puheista oikeiden osuus |
| tasapainotettu tarkkuus | balanced accuracy | puolueiden saantien keskiarvo; jokainen puolue painaa yhtä paljon. Arvaustaso 8 puolueella = 0,125 |
| F1 | F1 score | puolueen täsmällisyyden ja saannin harmoninen keskiarvo (0–1) |
| macro-F1 | macro-averaged F1 | puolueiden F1-arvojen tavallinen keskiarvo; kuten tasapainotettu tarkkuus, mutta rankaisee myös vääriä arvauksia puolueelle. Arvaustaso noin 0,125 |
| sekaannusmatriisi | confusion matrix | taulukko: rivi = oikea puolue, sarake = arvattu puolue, solu = osuus rivin puheista; perustuu mallin ykkösarvaukseen, ei ole symmetrinen |
| AUC | area under the ROC curve | todennäköisyys, että malli antaa satunnaiselle A:n puheelle suuremman pistemäärän kuin satunnaiselle B:n puheelle; 0,5 = arvaus |

**Tutkimuksen mittarit**

| käsite | englanniksi | merkitys |
|---|---|---|
| erottuvuus | (pairwise) separability | d = 2·AUC − 1 puolueparille; 0 = ei erotu, 1 = erottuu aina |
| erottuvuusmatriisi | separability matrix | symmetrinen puolueet × puolueet -taulukko, jonka solu on puolueparin erottuvuus; diagonaali 0. Puoluekartan pohja. Eri asia kuin sekaannusmatriisi |
| polarisaatio | polarisation | erottuvuuden keskiarvo kaikkien 28 puolueparin yli |
| edustajan profiili | MP profile | mallin todennäköisyyksien keskiarvo edustajan 20 puheesta |
| (näkemysten) monimuotoisuus | (within-party viewpoint) diversity | profiilien kohinakorjattu hajonta puolueen keskipisteen ympärillä |
| kohinakorjattu hajonta | split-half (noise-corrected) spread | hajonta, joka lasketaan kahden erillisistä puheista tehdyn profiilin tulona, jolloin satunnaisvaihtelu kumoutuu |
| suhteellinen monimuotoisuus | relative diversity | monimuotoisuus jaettuna puolueiden keskipisteiden keskimääräisellä etäisyydellä |
| leave-out-mittari, π | leave-out estimator, partisanship π | Gentzkowin ym. mittari: todennäköisyys arvata puhujan puolue oikein yhden sanaparin perusteella (0,5 = ei eroa); puolueen sanankäyttö lasketaan ilman arvioitavaa edustajaa |
| plug-in-mittari | plug-in estimator | sama ilman leave-out-korjausta; yliarvioi erot pienissä aineistoissa |
| MDS | multidimensional scaling | puolueiden sijoittaminen tasoon niin, että etäisyydet vastaavat erottuvuutta |
| pääkomponentti | principal component (PCA) | aineiston suurimman vaihtelun suunta; käytetään edustajakartassa |
| log-odds informatiivisella priorilla | weighted log-odds ratio with informative Dirichlet prior | sanan puoluesidonnaisuuden mittari, joka ei liioittele harvinaisia sanoja (EDA) |

**Tilastollinen tausta.** Lisää luettavaa kurssin *Statistics for Data Science* verkkomateriaalissa (SfDS):
- logistinen regressio: [SfDS 9.5](https://lacerbi.github.io/stats-for-ds-website/chapters/09-linear-logistic-regression.html#logistic-regression); mallin sovitus ristientropialla on suurimman uskottavuuden estimointia: [SfDS 6.7](https://lacerbi.github.io/stats-for-ds-website/chapters/06-parametric-inference-II.html#connection-to-machine-learning-cross-entropy-as-mle)
- ristiinvalidointi ja ylisovittaminen: [SfDS 9.4.4](https://lacerbi.github.io/stats-for-ds-website/chapters/09-linear-logistic-regression.html#model-selection-choosing-the-right-predictors)
- satunnaistesti (*permutation test*): [SfDS 7.5.1](https://lacerbi.github.io/stats-for-ds-website/chapters/07-hypothesis-testing.html#the-permutation-test-a-simulation-approach)
- bootstrap ja sen rajoitukset (miksi käytämme osaotantaa): [SfDS 4.5](https://lacerbi.github.io/stats-for-ds-website/chapters/04-nonparametric-bootstrap.html#the-bootstrap-simulating-uncertainty) ja [4.8](https://lacerbi.github.io/stats-for-ds-website/chapters/04-nonparametric-bootstrap.html#when-the-bootstrap-fails)
- kohinakorjattu hajonta: riippumattomien muuttujien kovarianssi on nolla, [SfDS 2.7.2](https://lacerbi.github.io/stats-for-ds-website/chapters/02-expectation.html#properties-of-covariance-and-correlation)

In [ ]:
import os
import sys
import json
from itertools import combinations

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline

sys.path.insert(0, "../src")
import config

KANSIO = "../Data/processed/"
TULOKSET = "../results/"
KAUDET = ["2015-2019", "2019-2023", "2023-2027"]
SIEMENET = {"2015-2019": 100, "2019-2023": 101, "2023-2027": 102}   # satunnaislukujen siemenet

# Asetukset
MIN_FRAASEJA = 20      # puheet, joissa on vähemmän sanaston fraaseja, jätetään pois
N_OPETUS = 300         # opetuspuheita per puolue per osio
OSIOITA = 5            # ristiinvalidoinnin osiot
EDUSTAJA_OSUUS = 0.8   # osuus kunkin puolueen edustajista yhdessä toistossa
PROFIILIN_PUHEET = 20  # puheita per edustajaprofiili (kaksi profiilia -> vähintään 40 puhetta)
C = 2.0                # regularisointi (valittu kaudella 2019-23, ks. lopun viritys)
TOISTOT = 8            # toistot oikeilla nimikkeillä
SATUNNAISET = 4        # toistot sekoitetuilla nimikkeillä
UUDELLEEN = False      # True: laske ajot uudelleen, vaikka tulokset ovat jo kansiossa results/

In [2]:
X_kaikki = sp.load_npz(KANSIO + "counts.npz").tocsr()
indeksi = pd.read_parquet(KANSIO + "counts_index.parquet")
fraaseja = np.asarray(X_kaikki.sum(axis=1)).ravel()
print("Matriisi:", X_kaikki.shape[0], "puhetta ×", X_kaikki.shape[1], "fraasia")

Matriisi: 130965 puhetta × 26203 fraasia


## Funktiot

Jokainen funktio tekee yhden vaiheen. Kaikki satunnaisuus tulee yhdestä satunnaislukugeneraattorista (`rng`) per kausi, joten tulokset toistuvat samoina.

In [3]:
def tee_malli():
    """TF-IDF + logistinen regressio."""
    return make_pipeline(TfidfTransformer(sublinear_tf=True), LogisticRegression(C=C, max_iter=3000))


def parien_erottuvuus(oikeat, todnak, luokat):
    """Erottuvuus d = 2*AUC - 1 jokaiselle luokkaparille."""
    log_p = np.log(np.clip(todnak, 1e-12, 1))
    tulos = {}
    for i, j in combinations(range(len(luokat)), 2):
        rivit = (oikeat == luokat[i]) | (oikeat == luokat[j])
        if rivit.sum() == 0 or len(np.unique(oikeat[rivit])) < 2:
            continue
        pisteet = log_p[rivit, i] - log_p[rivit, j]
        auc = roc_auc_score(oikeat[rivit] == luokat[i], pisteet)
        tulos[(luokat[i], luokat[j])] = 2 * auc - 1
    return tulos


def edustajaprofiilit(puheet, todnak, luokat, rng):
    """Kaksi profiilia per edustaja: todennäköisyyksien keskiarvo kahdesta erillisestä 20 puheen joukosta."""
    k = PROFIILIN_PUHEET
    rivit = []
    for (henkilo, puolue), ryhma in puheet.groupby(["person_id", "label"]):
        if len(ryhma) < 2 * k:
            continue
        valitut = rng.choice(ryhma.index.values, size=2 * k, replace=False)
        profiili_a = todnak[puheet.index.get_indexer(valitut[:k])].mean(axis=0)
        profiili_b = todnak[puheet.index.get_indexer(valitut[k:])].mean(axis=0)
        rivi = {"person_id": henkilo, "label": puolue, "n_speeches": len(ryhma)}
        for n, luokka in enumerate(luokat):
            rivi["p_" + luokka] = (profiili_a[n] + profiili_b[n]) / 2
            rivi["a_" + luokka] = profiili_a[n]
            rivi["b_" + luokka] = profiili_b[n]
        rivit.append(rivi)
    return pd.DataFrame(rivit)


def monimuotoisuus(profiilit, luokat):
    """Puolueen sisäinen hajonta: naiivi, kohinakorjattu ja suhteellinen."""
    P = profiilit[["p_" + l for l in luokat]].values
    A = profiilit[["a_" + l for l in luokat]].values
    B = profiilit[["b_" + l for l in luokat]].values

    keskipisteet = {}
    for luokka in luokat:
        rivit = (profiilit["label"] == luokka).values
        if rivit.sum() >= 1:
            keskipisteet[luokka] = P[rivit].mean(axis=0)
    etaisyydet = [np.linalg.norm(keskipisteet[a] - keskipisteet[b]) for a, b in combinations(keskipisteet, 2)]
    puolueiden_valinen = np.mean(etaisyydet)

    tulos = {}
    for luokka in luokat:
        rivit = (profiilit["label"] == luokka).values
        if rivit.sum() < 3:
            continue
        naiivi = np.linalg.norm(P[rivit] - P[rivit].mean(axis=0), axis=1).mean()
        kovarianssi = np.mean(np.sum((A[rivit] - A[rivit].mean(axis=0)) * (B[rivit] - B[rivit].mean(axis=0)), axis=1))
        korjattu = np.sqrt(max(kovarianssi, 0.0))
        tulos[luokka] = {
            "diversity_naive": float(naiivi),
            "diversity": float(korjattu),
            "diversity_relative": float(korjattu / puolueiden_valinen),
            "own_prob": float(P[rivit, luokat.index(luokka)].mean()),
            "n_mps_profiled": int(rivit.sum()),
        }
    return tulos

In [4]:
def yleisin(arvot):
    """Yleisin arvo (edustajan puolue, jos hän on vaihtanut puoluetta kauden aikana)."""
    return arvot.mode().iat[0]


def yksi_toisto(X, puheet, luokat, rng, sekoita):
    """Yksi toisto: arvotaan 80 % edustajista, ristiinvalidointi edustajittain,
    testiennusteet jokaiselle testiedustajan puheelle."""
    if sekoita:
        puheet = puheet.copy()
        # jokainen edustaja saa yhden nimikkeen; nimikkeet sekoitetaan edustajien kesken
        edustajan_puolue = puheet.groupby("person_id")["label"].agg(yleisin)
        sekoitettu = pd.Series(rng.permutation(edustajan_puolue.values), index=edustajan_puolue.index)
        puheet["label"] = puheet["person_id"].map(sekoitettu).values

    # arvotaan 80 % kunkin puolueen edustajista
    valitut = set()
    for luokka in luokat:
        edustajat = puheet.loc[puheet["label"] == luokka, "person_id"].unique()
        maara = max(2, int(round(EDUSTAJA_OSUUS * len(edustajat))))
        valitut.update(rng.choice(edustajat, size=min(maara, len(edustajat)), replace=False))
    rivit = puheet["person_id"].isin(valitut).values
    puheet = puheet[rivit].copy()
    X = X[rivit]
    oikeat = puheet["label"].values
    ryhmat = puheet["person_id"].values

    todnak = np.full((len(puheet), len(luokat)), np.nan)
    jako = StratifiedGroupKFold(n_splits=OSIOITA, shuffle=True, random_state=int(rng.integers(1e9)))
    for opetus, testi in jako.split(X, oikeat, ryhmat):
        # tasapainotettu opetusaineisto: N_OPETUS puhetta per puolue
        opetusrivit = []
        for luokka in luokat:
            puolueen_rivit = opetus[oikeat[opetus] == luokka]
            if len(puolueen_rivit) == 0:
                continue
            if len(puolueen_rivit) < N_OPETUS:
                print(f"    huom: vain {len(puolueen_rivit)} opetuspuhetta puolueelle {luokka}")
            maara = min(N_OPETUS, len(puolueen_rivit))
            opetusrivit.extend(rng.choice(puolueen_rivit, size=maara, replace=False))
        opetusrivit = np.array(opetusrivit)

        malli = tee_malli()
        malli.fit(X[opetusrivit], oikeat[opetusrivit])
        ennuste = malli.predict_proba(X[testi])

        # sarakkeet samaan järjestykseen kuin luokat (osiosta voi puuttua luokka)
        taysi = np.zeros((len(testi), len(luokat)))
        for n, luokka in enumerate(malli.classes_):
            taysi[:, luokat.index(luokka)] = ennuste[:, n]
        todnak[testi] = taysi

    return puheet.reset_index(drop=True), todnak


def arvioi(puheet, todnak, luokat, rng):
    """Kaikki mittarit yhdestä toistosta."""
    oikeat = puheet["label"].values
    arvaus = np.array(luokat)[todnak.argmax(axis=1)]
    tulos = {
        "accuracy": accuracy_score(oikeat, arvaus),
        "balanced_accuracy": balanced_accuracy_score(oikeat, arvaus),
        "macro_f1": f1_score(oikeat, arvaus, average="macro"),
        "n_speeches": len(oikeat),
        "n_mps": puheet["person_id"].nunique(),
    }
    erottuvuus = parien_erottuvuus(oikeat, todnak, luokat)
    tulos["polarisation"] = float(np.mean(list(erottuvuus.values())))
    sekaannus = confusion_matrix(oikeat, arvaus, labels=luokat, normalize="true")
    profiilit = edustajaprofiilit(puheet, todnak, luokat, rng)
    hajonta = monimuotoisuus(profiilit, luokat)
    return tulos, erottuvuus, sekaannus, profiilit, hajonta

In [5]:
def aja_kausi(X, indeksi, kausi, luokat, luokkasarake, toistot, satunnaiset):
    """Kaikki toistot yhdelle kaudelle."""
    rng = np.random.default_rng(SIEMENET[kausi])
    rivit = (indeksi["term"] == kausi).values & indeksi[luokkasarake].isin(luokat).values
    rivit = rivit & (np.asarray(X.sum(axis=1)).ravel() >= MIN_FRAASEJA)
    puheet = indeksi[rivit].reset_index(drop=True).copy()
    puheet["label"] = puheet[luokkasarake].values
    X_kausi = X[rivit]

    yhteenveto, parit, sekaannukset, profiilit, hajonnat, ennusteet = [], [], [], [], [], []
    for toisto in range(toistot + satunnaiset):
        sekoita = toisto >= toistot
        sarja = "random" if sekoita else "real"
        osa, todnak = yksi_toisto(X_kausi, puheet, luokat, rng, sekoita)
        tulos, erottuvuus, sekaannus, profiili, hajonta = arvioi(osa, todnak, luokat, rng)

        yhteenveto.append({"term": kausi, "series": sarja, "rep": toisto, **tulos})
        for (a, b), d in erottuvuus.items():
            parit.append({"term": kausi, "series": sarja, "rep": toisto, "a": a, "b": b, "d": d})
        if not sekoita:
            sekaannukset.append(sekaannus)
            profiili["term"] = kausi
            profiili["rep"] = toisto
            profiilit.append(profiili)
            # jokaisen testipuheen ennuste (muistio 11)
            ennuste = pd.DataFrame(todnak.astype("float32"), columns=["p_" + l for l in luokat])
            ennuste.insert(0, "speech_id", osa["speech_id"].values)
            ennuste.insert(1, "label", osa["label"].values)
            ennuste.insert(2, "rep", toisto)
            ennusteet.append(ennuste)
        for puolue, arvot in hajonta.items():
            hajonnat.append({"term": kausi, "series": sarja, "rep": toisto, "party": puolue, **arvot})
        print(f"  {kausi} {sarja:6s} toisto {toisto}: tasap. tarkkuus {tulos['balanced_accuracy']:.3f}, "
              f"polarisaatio {tulos['polarisation']:.3f}")

    sekaannus_ka = pd.DataFrame(np.mean(sekaannukset, axis=0), index=luokat, columns=luokat)
    return (yhteenveto, parit, sekaannus_ka, pd.concat(profiilit, ignore_index=True), hajonnat,
            pd.concat(ennusteet, ignore_index=True))


def aja(tunnus, X, indeksi, luokat_kaudelle, luokkasarake="party", kaudet=KAUDET, tallenna_ennusteet=False):
    """Ajaa kaikki kaudet ja tallentaa tulokset tiedostoihin results/<tunnus>_*.
    tallenna_ennusteet=True tallentaa myös puhekohtaiset testiennusteet (Data/processed/<tunnus>_oof.parquet).
    Jos tulokset ovat jo olemassa eikä UUDELLEEN ole True, ne luetaan tiedostosta eikä mallia ajeta."""
    tiedostot = [TULOKSET + tunnus + "_summary.csv"]
    if tallenna_ennusteet:
        tiedostot.append(KANSIO + tunnus + "_oof.parquet")
    if not UUDELLEEN and all(os.path.exists(t) for t in tiedostot):
        print(f"[{tunnus}] tulokset on jo laskettu, luetaan tiedostosta {tiedostot[0]}")
        tiivis = pd.read_csv(tiedostot[0]).groupby(["term", "series"])[["balanced_accuracy", "polarisation"]].mean()
        print(tiivis.round(3))
        return

    yhteenveto, parit, profiilit, hajonnat, sekaannukset, ennusteet = [], [], [], [], {}, []
    for kausi in kaudet:
        print(f"[{tunnus}] {kausi}")
        tulos = aja_kausi(X, indeksi, kausi, luokat_kaudelle(kausi), luokkasarake, TOISTOT, SATUNNAISET)
        yhteenveto.extend(tulos[0])
        parit.extend(tulos[1])
        sekaannukset[kausi] = tulos[2].round(4).to_dict()
        profiilit.append(tulos[3])
        hajonnat.extend(tulos[4])
        ennusteet.append(tulos[5].assign(term=kausi))

    pd.DataFrame(yhteenveto).to_csv(TULOKSET + tunnus + "_summary.csv", index=False)
    if all(len(luokat_kaudelle(kausi)) == 2 for kausi in kaudet):
        print("(kahden luokan ajo: vain _summary tallennetaan)")
        return
    pd.DataFrame(parit).to_csv(TULOKSET + tunnus + "_pairwise.csv", index=False)
    pd.concat(profiilit).to_csv(TULOKSET + tunnus + "_mp_profiles.csv", index=False)
    pd.DataFrame(hajonnat).to_csv(TULOKSET + tunnus + "_diversity.csv", index=False)
    with open(TULOKSET + tunnus + "_confusion.json", "w") as f:
        json.dump(sekaannukset, f, indent=1)
    if tallenna_ennusteet:
        pd.concat(ennusteet, ignore_index=True).to_parquet(KANSIO + tunnus + "_oof.parquet", index=False)

    tiivis = pd.DataFrame(yhteenveto).groupby(["term", "series"])[["balanced_accuracy", "polarisation"]].mean()
    print(tiivis.round(3))

## Ajo 1: päätulos, 8 puoluetta

Mallissa ovat puolueet, jotka ovat eduskunnassa kaikilla kausilla: VAS, SDP, VIHR, KESK, RKP, KD, KOK ja PS.

Jos pienen puolueen (KD, RKP) aktiivisimmat edustajat osuvat samaan testiryhmään, opetuspuheita jää hieman alle 300. Tuloste ilmoittaa nämä *huom*-riveinä; niitä on muutama.

In [6]:
def kahdeksan_puoluetta(kausi):
    return list(config.MAIN_PARTIES)


aja("party", X_kaikki, indeksi, kahdeksan_puoluetta, tallenna_ennusteet=True)

[party] 2015-2019
  2015-2019 real   toisto 0: tasap. tarkkuus 0.254, polarisaatio 0.406
  2015-2019 real   toisto 1: tasap. tarkkuus 0.244, polarisaatio 0.385
  2015-2019 real   toisto 2: tasap. tarkkuus 0.250, polarisaatio 0.401
  2015-2019 real   toisto 3: tasap. tarkkuus 0.247, polarisaatio 0.395
  2015-2019 real   toisto 4: tasap. tarkkuus 0.246, polarisaatio 0.377
  2015-2019 real   toisto 5: tasap. tarkkuus 0.236, polarisaatio 0.350
  2015-2019 real   toisto 6: tasap. tarkkuus 0.256, polarisaatio 0.410
  2015-2019 real   toisto 7: tasap. tarkkuus 0.261, polarisaatio 0.407
    huom: vain 241 opetuspuhetta puolueelle KD
  2015-2019 random toisto 8: tasap. tarkkuus 0.135, polarisaatio -0.022
  2015-2019 random toisto 9: tasap. tarkkuus 0.152, polarisaatio 0.151
  2015-2019 random toisto 10: tasap. tarkkuus 0.111, polarisaatio -0.091
  2015-2019 random toisto 11: tasap. tarkkuus 0.117, polarisaatio -0.052
[party] 2019-2023
  2019-2023 real   toisto 0: tasap. tarkkuus 0.279, polarisa

## Ajo 2: kontrollitehtävä, hallitus vs. oppositio

Sama menetelmä, mutta arvattavana on puhujan hallitusasema puheen hetkellä. Jos tämä erottuvuus kasvaa samaan tahtiin kuin puolueiden, puolue-erojen kasvu voi johtua hallitus–oppositio-asetelmasta.

In [7]:
indeksi["asema"] = indeksi["in_government"].map({True: "hallitus", False: "oppositio"})


def hallitus_ja_oppositio(kausi):
    return ["hallitus", "oppositio"]


aja("government", X_kaikki, indeksi, hallitus_ja_oppositio, luokkasarake="asema")

[government] 2015-2019
  2015-2019 real   toisto 0: tasap. tarkkuus 0.624, polarisaatio 0.350
  2015-2019 real   toisto 1: tasap. tarkkuus 0.620, polarisaatio 0.339
  2015-2019 real   toisto 2: tasap. tarkkuus 0.626, polarisaatio 0.354
  2015-2019 real   toisto 3: tasap. tarkkuus 0.637, polarisaatio 0.382
  2015-2019 real   toisto 4: tasap. tarkkuus 0.614, polarisaatio 0.322
  2015-2019 real   toisto 5: tasap. tarkkuus 0.622, polarisaatio 0.346
  2015-2019 real   toisto 6: tasap. tarkkuus 0.624, polarisaatio 0.350
  2015-2019 real   toisto 7: tasap. tarkkuus 0.618, polarisaatio 0.334
  2015-2019 random toisto 8: tasap. tarkkuus 0.491, polarisaatio -0.020
  2015-2019 random toisto 9: tasap. tarkkuus 0.501, polarisaatio 0.003
  2015-2019 random toisto 10: tasap. tarkkuus 0.491, polarisaatio -0.030
  2015-2019 random toisto 11: tasap. tarkkuus 0.498, polarisaatio -0.005
[government] 2019-2023
  2019-2023 real   toisto 0: tasap. tarkkuus 0.678, polarisaatio 0.478
  2019-2023 real   toisto 

## Ajo 3: ilman ministerien puheita

Ministerit pitävät noin 20–38 % hallituspuolueen puheista (EDA) ja puhuvat hallituksen edustajina. Jos tulos muuttuu paljon ilman heitä, puolue-erot ovat osin ministerien virkakieltä.

**Huom:** ilman ministerejä RKP:llä ja KD:llä on monessa osiossa alle 300 opetuspuhetta (tulosteen *huom*-rivit), koska suuri osa niiden puheista kaudella 2023–27 on ministerien. Näissä osioissa opetusaineisto ei ole tasapainossa, ja satunnaissarjan polarisaatio heittelee (2023–27: −0,12). Oikeiden nimikkeiden tulos on silti lähes sama kuin ajossa 1.

In [8]:
ei_ministeri = ~indeksi["is_minister"].to_numpy()
indeksi_ilman = indeksi[ei_ministeri].reset_index(drop=True)
X_ilman = X_kaikki[ei_ministeri]

aja("party_nominister", X_ilman, indeksi_ilman, kahdeksan_puoluetta)

[party_nominister] 2015-2019
  2015-2019 real   toisto 0: tasap. tarkkuus 0.251, polarisaatio 0.396
  2015-2019 real   toisto 1: tasap. tarkkuus 0.254, polarisaatio 0.392
  2015-2019 real   toisto 2: tasap. tarkkuus 0.253, polarisaatio 0.411
  2015-2019 real   toisto 3: tasap. tarkkuus 0.244, polarisaatio 0.380
  2015-2019 real   toisto 4: tasap. tarkkuus 0.256, polarisaatio 0.419
  2015-2019 real   toisto 5: tasap. tarkkuus 0.241, polarisaatio 0.373
  2015-2019 real   toisto 6: tasap. tarkkuus 0.247, polarisaatio 0.376
  2015-2019 real   toisto 7: tasap. tarkkuus 0.238, polarisaatio 0.351
  2015-2019 random toisto 8: tasap. tarkkuus 0.141, polarisaatio 0.072
  2015-2019 random toisto 9: tasap. tarkkuus 0.126, polarisaatio -0.024
  2015-2019 random toisto 10: tasap. tarkkuus 0.137, polarisaatio 0.057
  2015-2019 random toisto 11: tasap. tarkkuus 0.115, polarisaatio -0.055
[party_nominister] 2019-2023
    huom: vain 278 opetuspuhetta puolueelle RKP
    huom: vain 272 opetuspuhetta puolu

## Ajo 4: Siniset mukana (2015–2019)

Siniset erosivat perussuomalaisista kesäkuussa 2017. Kuinka lähellä ne ovat puheissaan PS:ää ja Kokoomusta?

In [9]:
def yhdeksan_puoluetta(kausi):
    return list(config.PARTY_ORDER)


aja("party_blue", X_kaikki, indeksi, yhdeksan_puoluetta, kaudet=["2015-2019"])

[party_blue] 2015-2019
  2015-2019 real   toisto 0: tasap. tarkkuus 0.232, polarisaatio 0.394
  2015-2019 real   toisto 1: tasap. tarkkuus 0.231, polarisaatio 0.401
  2015-2019 real   toisto 2: tasap. tarkkuus 0.230, polarisaatio 0.389
  2015-2019 real   toisto 3: tasap. tarkkuus 0.229, polarisaatio 0.394
  2015-2019 real   toisto 4: tasap. tarkkuus 0.242, polarisaatio 0.423
  2015-2019 real   toisto 5: tasap. tarkkuus 0.231, polarisaatio 0.400
  2015-2019 real   toisto 6: tasap. tarkkuus 0.227, polarisaatio 0.399
  2015-2019 real   toisto 7: tasap. tarkkuus 0.233, polarisaatio 0.410
  2015-2019 random toisto 8: tasap. tarkkuus 0.129, polarisaatio 0.096
  2015-2019 random toisto 9: tasap. tarkkuus 0.118, polarisaatio 0.027
    huom: vain 220 opetuspuhetta puolueelle KD
  2015-2019 random toisto 10: tasap. tarkkuus 0.103, polarisaatio -0.020
  2015-2019 random toisto 11: tasap. tarkkuus 0.123, polarisaatio 0.060
                  balanced_accuracy  polarisation
term      series         

## Ajo 5: ilman tyylisanoja

Muistion 10 sanalistoissa monen puolueen painavimpia fraaseja ovat tyylisanat, kuten *kyl* (kyllä), *sit* (sitten), *tääl* (täällä), *tode* (todeta, todella) ja *tietyst* (tietysti). NLTK:n täytesanalistassa ei ole näitä yleisiä adverbeja ja partikkeleita. Jos puolueiden erottuvuus ja sen kasvu johtuvat vain puhetavasta, tulos muuttuu, kun nämä sanat poistetaan.

**Poistettavat fraasit** (kaikki sanat ja sanaparit, joissa jokin osa on listalla):
1. **Tyylisanat**: `src/stopwords_fi_style.txt`. Siinä on noin 130 sanaa kolmesta luokasta: puhekieliset muodot, diskurssipartikkelit ja vahvistimet sekä puhetekoverbit (*todeta*, *sanoa*, *ajatella*, *haluta*, *toivoa*). Lista laadittiin luokkien perusteella eikä painojen perusteella. Tiedoston lopussa ovat sanat, jotka jätettiin pois, koska niiden vartalo on yhteinen sisältösanan kanssa (esim. *kanssa* → *kans* = *kansa*).
2. **Puolueiden nimet, jotka jäivät siivouksessa sanastoon**: *sosiaalidemokraat\**, *demar*, *vasemmisto\** ja yksittäinen sana *vihreä* (sanapari *vihreä siirtymä* jää). Muistion 02 säännöllinen lauseke tunnistaa SDP:n nimestä vain puolueen virallisen kirjoitusasun *sosialidemokraat-* (noin 1 600 esiintymää puheissa), mutta ei yleisempää muotoa *sosiaalidemokraat-* (noin 4 600). Lauseke ei myöskään tunnista muotoja *vasemmisto* ja *demareiden*.

Sanat poistetaan sanastosta eli matriisin sarakkeista. Siivousta ei ajeta uudelleen, joten sanaparit, jotka olisivat syntyneet poistetun sanan yli, puuttuvat. Tämä on robustisuustarkistukselle riittävä likiarvo. Puheet, joihin jää alle 20 fraasia, putoavat pois, joten otos on hieman pienempi kuin ajossa 1.

Tulokset verrataan ajoon 1 muistiossa 07.

In [10]:
import Stemmer

vartaloija = Stemmer.Stemmer("finnish")
with open("../src/stopwords_fi_style.txt", encoding="utf-8") as f:
    tyylisanat = [sana for rivi in f if not rivi.startswith("#") for sana in rivi.split()]
tyylivartalot = set(vartaloija.stemWords(tyylisanat))

with open(KANSIO + "vocab.txt", encoding="utf-8") as f:
    sanasto = np.array([rivi.strip() for rivi in f if rivi.strip()])


def poistetaanko(fraasi):
    osat = fraasi.split("_")
    tyylia = any(osa in tyylivartalot for osa in osat)
    puolueen_nimi = any(osa.startswith(("sosiaalidemokraat", "demar", "vasemmisto")) for osa in osat)
    return tyylia or puolueen_nimi or fraasi == "vihreä"


poistettavat = np.array([poistetaanko(fraasi) for fraasi in sanasto])
print("Tyylivartaloita:", len(tyylivartalot), " poistettavia fraaseja:", poistettavat.sum(), "/", len(sanasto))
print("Esimerkkejä:", ", ".join(sanasto[poistettavat][:40]))

X_tyyliton = X_kaikki[:, ~poistettavat]
fraaseja_ennen = (fraaseja >= MIN_FRAASEJA).sum()
fraaseja_jalkeen = (np.asarray(X_tyyliton.sum(axis=1)).ravel() >= MIN_FRAASEJA).sum()
print("Puheita, joissa vähintään", MIN_FRAASEJA, "fraasia:", fraaseja_ennen, "->", fraaseja_jalkeen)
print("Osuus fraasien esiintymistä, joka poistuu:", round(1 - X_tyyliton.sum() / X_kaikki.sum(), 3))

aja("party_nostyle", X_tyyliton, indeksi, kahdeksan_puoluetta, tallenna_ennusteet=True)

Tyylivartaloita: 114  poistettavia fraaseja: 3713 / 26203
Esimerkkejä: aiem_jo, aiem_tote, aiem_tääl, aiheut_myös, aika_aina, aika_aiva, aika_eli, aika_esimerk, aika_jo, aika_juuri, aika_kuite, aika_kyl, aika_myös, aika_myösk, aika_näin, aika_sano, aika_siel, aika_siis, aika_sit, aika_tava, aika_tode, aika_tääl, aika_vielä, aikais_jo, aikais_sano, aikais_tote, aikais_tääl, aiko_myös, aiko_sit, aina, aina_aika, aina_en, aina_hallituks, aina_helpo, aina_hyv, aina_hyvä, aina_its, aina_joku, aina_kaik, aina_kaks
Puheita, joissa vähintään 20 fraasia: 130151 -> 129928
Osuus fraasien esiintymistä, joka poistuu: 0.123
[party_nostyle] 2015-2019
  2015-2019 real   toisto 0: tasap. tarkkuus 0.252, polarisaatio 0.407
  2015-2019 real   toisto 1: tasap. tarkkuus 0.234, polarisaatio 0.353
  2015-2019 real   toisto 2: tasap. tarkkuus 0.257, polarisaatio 0.415
  2015-2019 real   toisto 3: tasap. tarkkuus 0.246, polarisaatio 0.395
  2015-2019 real   toisto 4: tasap. tarkkuus 0.239, polarisaatio 0.367
 

## Liite: regularisoinnin valinta

Regularisointiparametri C valittiin kerran kaudella 2019–23 (yksi toisto, ei satunnaistoistoja). Tarkkuus oli lähes sama koko välillä:

| C | 0,5 | 1 | 2 | 5 | 10 | 20 |
|---|---|---|---|---|---|---|
| tasapainotettu tarkkuus | 0,273 | 0,276 | 0,278 | 0,277 | 0,277 | 0,277 |

Valinnan voi toistaa asettamalla `VIRITA = True`.

In [ ]:
VIRITA = False

if VIRITA:
    for kokeiltava in [0.5, 1, 2, 5, 10, 20]:
        C = kokeiltava
        tulos = aja_kausi(X_kaikki, indeksi, "2019-2023", list(config.MAIN_PARTIES), "party", 1, 0)
        print("C =", kokeiltava, "tasapainotettu tarkkuus:", round(tulos[0][0]["balanced_accuracy"], 3))
    C = 2.0

  2019-2023 real   toisto 0: tasap. tarkkuus 0.278, polarisaatio 0.448
C = 0.5 tasapainotettu tarkkuus: 0.278
  2019-2023 real   toisto 0: tasap. tarkkuus 0.277, polarisaatio 0.455
C = 1 tasapainotettu tarkkuus: 0.277
  2019-2023 real   toisto 0: tasap. tarkkuus 0.279, polarisaatio 0.461
C = 2 tasapainotettu tarkkuus: 0.279
  2019-2023 real   toisto 0: tasap. tarkkuus 0.277, polarisaatio 0.463
C = 5 tasapainotettu tarkkuus: 0.277
  2019-2023 real   toisto 0: tasap. tarkkuus 0.280, polarisaatio 0.463
C = 10 tasapainotettu tarkkuus: 0.28
  2019-2023 real   toisto 0: tasap. tarkkuus 0.281, polarisaatio 0.463
C = 20 tasapainotettu tarkkuus: 0.281
